# 12 · Play a predictive AHR candidate

Use this local notebook to inspect a trained V3 checkpoint on a clickable 9×9 Go board. The player is labelled **experimental candidate, not promoted**. Original notebook 12 and the selected baseline remain unchanged. The predictor improves training representations; latent-guided move ranking stays off here. AHR uses two Queen cycles and a one-second move budget, with exact Go legality and full positional-superko history.

**Inputs:** an explicit V3 training run and its latest checkpoint. **Outputs:** complete game, settings, actions, model weights, every RNG and undo records in `checkpoints/<play_session>/`; an atomic manifest retains latest two and best. Saving and loading reject changed checkpoints or definitions. One writer per game; a second stale widget must reload before editing.

**Run All:** by default, performs a short labelled untrained-fixture check and shows readiness; no trained model or service is started. Set `AHRV3_PLAY_RUN` to a trained run such as `ahr-v3-C-s01`, optionally `AHRV3_PLAY_SESSION` to a fresh simple game ID, then restart the kernel and Run All. First execution usually takes under one minute; each AI move has a one-second inference budget. No external engine or network service is used.

**Exact resume:** keep the same training run and play session, restart the kernel, and Run All with the same environment settings. The widget loads the latest valid complete checkpoint and restores game history and RNG. Undo removes the last human/AI pair (or available moves), restores the RNG before it, and saves a new revision. The saved game retains its pinned model weights even if training advances; use a new play-session ID to select a newer model. Corruption never silently starts a new game.

In [ ]:
from pathlib import Path
import nbformat,os
PROJECT=Path.cwd().resolve()
while not (PROJECT/'notebooks/00_instance_setup_and_recovery.ipynb').is_file():
    if PROJECT.parent==PROJECT:raise RuntimeError('Run inside the AHR Go project')
    PROJECT=PROJECT.parent
for cell in nbformat.read(PROJECT/'experiments/ahr_jepa_v3/00_shared.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,'00_shared:ahr-export','exec'),globals())
v3_load_upstream(PROJECT,globals())
for name in ('08_predictive_ahr.ipynb','09_trajectory_learning.ipynb'):
    v3_import_exports(PROJECT/V3_ROOT/name,globals())
torch.set_num_threads(1)

## A saved game and its frozen player

Moves are applied only after exact legality checks. A deadline, failed inference or illegal proposal is shown as UNKNOWN and leaves the board unchanged. The board can be reloaded after an interruption; the checkpoint includes the full chronological action record and every random generator. Changing color starts a new game only when New game is clicked.

In [ ]:
import ipywidgets as widgets
from IPython.display import display,HTML
import html,copy,re,time

V3_PLAY_SCHEMA='ahr_v3_local_play_v1'
V3_PLAY_RESUME='12_play_predictive_ahr: same AHRV3_PLAY_RUN and AHRV3_PLAY_SESSION; restart kernel and Run All'

def v3_play_sources(project):
    sources=v3_source_identity(project)
    path=Path(project)/V3_ROOT/'12_play_predictive_ahr.ipynb'
    sources[str(path.relative_to(Path(project)))]=v3_code_sha(path,True)
    return sources

def v3_play_reconstruct(actions,komi=7.5):
    board=initial_state(9,komi)
    for action in actions:board=play(board,action)
    return board

def v3_play_validate(snapshot,identity):
    if not isinstance(snapshot,dict) or REQUIRED_CHECKPOINT_KEYS-set(snapshot):raise RuntimeError('Incomplete saved game')
    if snapshot.get('schema')!=V3_PLAY_SCHEMA or snapshot.get('config')!=identity:raise RuntimeError('Saved game belongs to a different model, rules, sources or settings')
    if snapshot.get('identity_sha256')!=v3_digest(identity):raise RuntimeError('Saved game identity corrupted')
    if any(snapshot.get(key) is not None for key in ('optimizer','scheduler','scaler','replay')):raise RuntimeError('Play checkpoint contains unexpected training state')
    if snapshot['optimizer_steps']!=0 or snapshot['distributed']!={'world_size':1}:raise RuntimeError('Invalid play execution mode')
    if arena_tensor_digest(snapshot['model'])!=identity['model_weights_sha256']:raise RuntimeError('Frozen player weights changed')
    actions=snapshot['actions'];records=snapshot['turn_records']
    if len(records)!=len(actions) or any(r['action']!=a for r,a in zip(records,actions)):raise RuntimeError('Undo/action record mismatch')
    board=v3_play_reconstruct(actions,identity['komi'])
    if to_dict(board)!=snapshot['game'] or snapshot['sampler']['next_move']!=len(actions):raise RuntimeError('Chronological replay does not match saved full history')
    if snapshot['settings']['human_color'] not in (BLACK,WHITE):raise RuntimeError('Invalid human color')
    if snapshot['metrics']['terminal']!=board.terminal:raise RuntimeError('Terminal status mismatch')
    return True

class V3PlaySession:
    def __init__(self,project,run_id,session_id,*,device='cpu',selection='latest',fixture_model=None):
        self.project=Path(project);self.run_id=run_id;self.session_id=session_id
        if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]{0,90}',session_id):raise ValueError('Use a simple play session ID')
        self.directory=self.project/'checkpoints'/session_id
        with checkpoint_lock(self.directory):existing=arena_strict_load(self.directory)
        if fixture_model is None:
            if existing is not None:
                provenance=copy.deepcopy(existing['config']['provenance'])
                if provenance.get('kind')!='trained_v3_candidate' or provenance.get('run_id')!=run_id:raise ValueError('Saved game is not this trained candidate')
                if existing['config'].get('sources')!=v3_play_sources(self.project):raise RuntimeError('Saved player definitions changed')
                self.model=v3_model_from_config(provenance['model_config'],device)
                self.model.load_state_dict(existing['model'],strict=True)
            else:
                self.model,metadata=v3_load_model(self.project,run_id,device=device,selection=selection)
                if metadata['optimizer_steps']<1 or metadata['provenance'].get('purpose')!='go_training' or metadata['config'].get('validation_force_pass',False):raise ValueError('An explicitly trained, genuine V3 checkpoint is required; fixtures are not candidates')
                provenance={'kind':'trained_v3_candidate','run_id':run_id,'checkpoint_sha256':metadata['checkpoint_sha256'],
                    'checkpoint_file':metadata['checkpoint_file'],'optimizer_steps':metadata['optimizer_steps'],'model_config':metadata['model_config']}
            self.label='Experimental candidate, not promoted: '+run_id+' · '+provenance['checkpoint_sha256'][:12]
        else:
            if not session_id.startswith('v3play-fixture-') or run_id!='UNTRAINED_WIRING_FIXTURE':raise ValueError('Fixture models require explicit isolated fixture provenance')
            if not isinstance(fixture_model,V3AHR):raise TypeError('Expected a V3 AHR fixture model')
            self.model=fixture_model;self.label='Untrained wiring fixture — no playing-strength claim'
            provenance={'kind':'untrained_wiring_fixture','run_id':run_id,'checkpoint_sha256':None,'optimizer_steps':0,'model_config':self.model.v3_config}
        self.model.eval().requires_grad_(False)
        self.search=V3SearchConfig(queen_cycles=2,use_latent_ranking=False)
        self.identity={'schema':V3_PLAY_SCHEMA,'provenance':provenance,'rules':RULESET,'komi':7.5,
            'model_weights_sha256':arena_tensor_digest(self.model.state_dict()),'search':asdict(self.search),
            'time_budget_s':1.,'sources':v3_play_sources(self.project),'device':str(next(self.model.parameters()).device)}
        self.directory=self.project/'checkpoints'/session_id
        with checkpoint_lock(self.directory):
            snapshot=arena_strict_load(self.directory)
            if snapshot is None:
                board=initial_state(9,7.5)
                snapshot={'schema':V3_PLAY_SCHEMA,'config':self.identity,'identity_sha256':v3_digest(self.identity),
                    'model':arena_cpu(self.model.state_dict()),'optimizer':None,'scheduler':None,'scaler':None,
                    'rng':capture_rng(),'sampler':{'next_move':0},'global_step':0,'metrics':{'terminal':False,'moves':0},
                    'optimizer_steps':0,'replay':None,'distributed':{'world_size':1},'game':to_dict(board),
                    'actions':[],'turn_records':[],'settings':{'human_color':BLACK,'auto_reply':True},
                    'last_error':None,'last_decision':None,'resets':0}
                v3_play_validate(snapshot,self.identity)
                _save_checkpoint_unlocked(self.directory,snapshot,best=True,resume_cell=V3_PLAY_RESUME)
            else:v3_play_validate(snapshot,self.identity)
            self.state=snapshot
        self.model.load_state_dict(self.state['model'],strict=True);restore_rng(self.state['rng'])

    @property
    def board(self):return from_dict(self.state['game'])

    def _current_locked(self):
        current=arena_strict_load(self.directory);v3_play_validate(current,self.identity)
        if current['global_step']!=self.state['global_step']:raise RuntimeError('Another widget saved this game; click Reload before making a move')
        if arena_tensor_digest(self.model.state_dict())!=self.identity['model_weights_sha256']:raise RuntimeError('Player weights changed during play')
        restore_rng(current['rng']);return current

    def _commit_locked(self,state):
        state['global_step']+=1;state['rng']=capture_rng();state['model']=arena_cpu(self.model.state_dict())
        state['sampler']['next_move']=len(state['actions'])
        board=from_dict(state['game']);state['metrics']={'moves':len(state['actions']),'terminal':board.terminal,'black_area_score':area_score(board) if board.terminal else None}
        v3_play_validate(state,self.identity)
        _save_checkpoint_unlocked(self.directory,state,best=board.terminal,resume_cell=V3_PLAY_RESUME)
        self.state=state;return state

    def reload(self):
        with checkpoint_lock(self.directory):
            state=arena_strict_load(self.directory);v3_play_validate(state,self.identity);self.state=state
        self.model.load_state_dict(state['model'],strict=True);restore_rng(state['rng']);return state

    def human_move(self,action):
        with checkpoint_lock(self.directory):
            state=self._current_locked();board=from_dict(state['game'])
            if board.terminal:raise ValueError('The game is finished; start a new game or undo')
            if board.to_play!=state['settings']['human_color']:raise ValueError('It is the AHR player’s turn')
            before=capture_rng();child=play(board,action)
            if arena_reference_move(board,action)!=child.board:raise RuntimeError('Independent legality verification failed')
            state['actions'].append(action);state['turn_records'].append({'actor':'human','action':action,'rng_before':before,'legal':True})
            state['game']=to_dict(child);state['last_error']=None;state['last_decision']=None
            return self._commit_locked(state)

    def ai_move(self):
        with checkpoint_lock(self.directory):
            state=self._current_locked();board=from_dict(state['game'])
            if board.terminal:raise ValueError('The game is finished')
            if board.to_play==state['settings']['human_color']:raise ValueError('It is the human’s turn')
            before=capture_rng();failure=None;result={};device=next(self.model.parameters()).device
            if device.type=='cuda':torch.cuda.synchronize(device)
            started=time.perf_counter()
            try:
                with torch.inference_mode():result=v3_choose_move(board,self.model,time_budget_s=1.,search_config=self.search)
                if device.type=='cuda':torch.cuda.synchronize(device)
            except Exception as error:failure='UNKNOWN_PLAYER_ERROR: '+type(error).__name__+': '+str(error)[:160]
            elapsed=time.perf_counter()-started;action=result.get('action')
            if elapsed>1. or result.get('deadline_missed'):failure='UNKNOWN_DEADLINE'
            if failure is None:
                try:
                    child=play(board,action)
                    if arena_reference_move(board,action)!=child.board:raise ValueError('Independent verifier disagreed')
                except (IllegalMove,TypeError,ValueError):failure='UNKNOWN_ILLEGAL_MOVE'
            audit={'action':action if type(action)is int else None,'elapsed_s':elapsed,'budget_s':1.,'queen_cycles':2,
                'nodes':result.get('nodes'),'model_calls':result.get('model_calls'),'status':failure or 'LEGAL_MOVE',
                'strategic_status':'UNKNOWN','checkpoint_io_in_timing':False,'latent_guidance':False}
            state['last_decision']=audit;state['last_error']=failure
            if failure is None:
                state['actions'].append(action);state['turn_records'].append({'actor':'AHR','action':action,'rng_before':before,'legal':True,'audit':audit})
                state['game']=to_dict(child)
            else:restore_rng(before)
            self._commit_locked(state)
            if failure:raise RuntimeError(failure+'; board unchanged. Retry AHR or undo.')
            return state

    def undo(self,plies=2):
        if type(plies)is not int or plies<1:raise ValueError('Undo needs a positive number of plies')
        with checkpoint_lock(self.directory):
            state=self._current_locked();count=min(plies,len(state['actions']))
            if not count:return state
            target=len(state['actions'])-count;restore_rng(state['turn_records'][target]['rng_before'])
            state['actions']=state['actions'][:target];state['turn_records']=state['turn_records'][:target]
            state['game']=to_dict(v3_play_reconstruct(state['actions'],self.identity['komi']))
            state['last_error']=None;state['last_decision']=None
            return self._commit_locked(state)

    def new_game(self,human_color=BLACK):
        if human_color not in (BLACK,WHITE):raise ValueError('Choose Black or White')
        with checkpoint_lock(self.directory):
            state=self._current_locked();state['game']=to_dict(initial_state(9,self.identity['komi']))
            state['actions']=[];state['turn_records']=[];state['settings']['human_color']=human_color
            state['last_error']=None;state['last_decision']=None;state['resets']+=1
            return self._commit_locked(state)

    def set_auto_reply(self,enabled):
        with checkpoint_lock(self.directory):
            state=self._current_locked();state['settings']['auto_reply']=bool(enabled)
            return self._commit_locked(state)


## Click to play

Choose an empty legal intersection or Pass. The AHR reply is automatic by default. Undo restores the previous human/AI pair, including random state. Reload reconciles another open widget with the saved game. Two legal passes end the game and display the exact project area score.

In [ ]:
class V3PlayWidget:
    def __init__(self,session):
        self.session=session;self.busy=False;self._buttons=[];self._rendering=False
        self.title=widgets.HTML();self.status=widgets.HTML();self.message=widgets.HTML()
        self.board_box=widgets.GridBox(layout=widgets.Layout(grid_template_columns='repeat(10, 40px)',grid_gap='1px'))
        self.pass_button=widgets.Button(description='Pass',icon='forward',layout=widgets.Layout(width='100px'))
        self.ai_button=widgets.Button(description='AHR move',icon='play',layout=widgets.Layout(width='120px'))
        self.undo_button=widgets.Button(description='Undo pair',icon='undo',layout=widgets.Layout(width='120px'))
        self.reload_button=widgets.Button(description='Reload',icon='refresh',layout=widgets.Layout(width='100px'))
        self.new_button=widgets.Button(description='New game',icon='plus',layout=widgets.Layout(width='120px'))
        self.color=widgets.Dropdown(options=[('Human: Black',BLACK),('Human: White',WHITE)],value=session.state['settings']['human_color'],layout=widgets.Layout(width='180px'))
        self.auto=widgets.Checkbox(value=session.state['settings']['auto_reply'],description='Automatic AHR reply',indent=False)
        self.pass_button.on_click(lambda _:self._perform(lambda:self._human(session.board.pass_action)))
        self.ai_button.on_click(lambda _:self._perform(session.ai_move))
        self.undo_button.on_click(lambda _:self._perform(session.undo))
        self.reload_button.on_click(lambda _:self._perform(session.reload))
        self.new_button.on_click(lambda _:self._perform(self._new_game))
        self.auto.observe(self._auto_changed,names='value')
        self.root=widgets.VBox([self.title,self.status,self.board_box,
            widgets.HBox([self.pass_button,self.ai_button,self.undo_button,self.reload_button]),
            widgets.HBox([self.color,self.new_button]),self.auto,self.message])
        self.render()

    def _human(self,action):
        self.session.human_move(action)
        if self.session.state['settings']['auto_reply'] and not self.session.board.terminal:self.session.ai_move()

    def _new_game(self):
        self.session.new_game(self.color.value)
        if self.session.state['settings']['auto_reply'] and self.color.value==WHITE:self.session.ai_move()

    def _auto_changed(self,change):
        if not self._rendering:self._perform(lambda:self.session.set_auto_reply(change['new']))

    def _perform(self,function):
        if self.busy:return
        self.busy=True;self.message.value='<span>Thinking / saving…</span>';self._set_disabled(True)
        try:
            function();self.message.value=''
        except Exception as error:self.message.value='<span style="color:#a32323">'+html.escape(str(error))+'</span>'
        finally:self.busy=False;self.render()

    def _board_click(self,action,revision):
        if revision!=self.session.state['global_step']:
            self.message.value='The board changed; use the current board.';return
        self._perform(lambda:self._human(action))

    def _set_disabled(self,disabled):
        for button in self._buttons:button.disabled=disabled or button.disabled
        for button in (self.pass_button,self.ai_button,self.undo_button,self.reload_button,self.new_button):button.disabled=disabled
        self.color.disabled=disabled;self.auto.disabled=disabled

    def render(self):
        self._rendering=True
        try:
            session=self.session;state=session.state;board=session.board;human=state['settings']['human_color']
            self.title.value='<h3 style="margin:4px 0">9×9 AHR Go</h3><div>'+html.escape(session.label)+'</div>'
            turn='Black' if board.to_play==BLACK else 'White';side='Black' if human==BLACK else 'White'
            if board.terminal:
                score=area_score(board);result='Draw' if score==0 else ('Black' if score>0 else 'White')+' wins by '+f'{abs(score):g}'
                text=f'Game finished · {result} · Area scoring, komi {board.komi:g}'
            else:text=f'Move {board.move_number} · {turn} to play · Human: {side} · Komi {board.komi:g}'
            self.status.value='<div style="padding:8px 0;font-weight:600">'+html.escape(text)+'</div>'
            legal=set(legal_actions(board)) if not board.terminal and board.to_play==human else set()
            revision=state['global_step'];last=state['actions'][-1] if state['actions'] else None
            old=self._buttons;self._buttons=[];items=[widgets.HTML('')]
            items.extend(widgets.HTML('<b style="display:block;text-align:center">'+c+'</b>') for c in 'ABCDEFGHJ')
            for row in range(9):
                items.append(widgets.HTML('<b style="display:block;padding-top:10px;text-align:center">'+str(9-row)+'</b>'))
                for col in range(9):
                    action=row*9+col;stone=board.board[action];mark='●' if stone==BLACK else '○' if stone==WHITE else '·'
                    button=widgets.Button(description=mark,tooltip='ABCDEFGHJ'[col]+str(9-row),disabled=action not in legal,
                        layout=widgets.Layout(width='40px',height='40px',border='2px solid #247ab5' if action==last else '1px solid #b58d59'),
                        style={'button_color':'#e9c68a','font_size':'27px','font_weight':'bold','text_color':'#171717'})
                    button.on_click(lambda _,a=action,r=revision:self._board_click(a,r))
                    self._buttons.append(button);items.append(button)
            self.board_box.children=items
            for button in old:button.close()
            self.pass_button.disabled=board.terminal or board.to_play!=human
            self.ai_button.disabled=board.terminal or board.to_play==human
            self.undo_button.disabled=not state['actions'];self.reload_button.disabled=False;self.new_button.disabled=False
            self.color.disabled=False;self.auto.disabled=False;self.auto.value=state['settings']['auto_reply']
            if state.get('last_error') and not self.message.value:self.message.value='<span style="color:#a32323">'+html.escape(state['last_error'])+'</span>'
        finally:self._rendering=False
        return self.root

    def display(self):display(self.root)


## Validate interaction and recovery

The bounded check below creates an isolated, explicitly untrained fixture. It clicks a legal board button, verifies the AHR reply, restores a second session from disk, undoes the pair with exact RNG restoration, rejects a stale writer, and verifies the two-pass terminal condition. Fixture results are never playing-strength evidence.

In [ ]:
def v3_play_nested_equal(left,right):
    if torch.is_tensor(left):return torch.is_tensor(right) and torch.equal(left,right)
    if isinstance(left,np.ndarray):return isinstance(right,np.ndarray) and np.array_equal(left,right)
    if isinstance(left,dict):return isinstance(right,dict) and set(left)==set(right) and all(v3_play_nested_equal(v,right[k]) for k,v in left.items())
    if isinstance(left,(list,tuple)):return type(left)is type(right) and len(left)==len(right) and all(v3_play_nested_equal(a,b) for a,b in zip(left,right))
    return left==right

def v3_play_smoke(project):
    v3_seed(2026092812);model=V3AHR().eval().requires_grad_(False)
    tag=uuid.uuid4().hex[:10];session_id='v3play-fixture-'+tag
    session=V3PlaySession(project,'UNTRAINED_WIRING_FIXTURE',session_id,fixture_model=model)
    initial_rng=copy.deepcopy(session.state['rng']);widget=V3PlayWidget(session)
    widget._buttons[0].click()
    if len(session.state['actions'])!=2:raise AssertionError('Real board click plus AHR reply failed: '+str(session.state['last_error']))
    assert session.state['actions'][0]==0 and session.state['last_decision']['queen_cycles']==2
    expected=copy.deepcopy(session.state)
    restored=V3PlaySession(project,'UNTRAINED_WIRING_FIXTURE',session_id,fixture_model=copy.deepcopy(model))
    assert restored.state['game']==expected['game'] and restored.state['actions']==expected['actions']
    assert v3_play_nested_equal(capture_rng(),expected['rng'])
    widget.undo_button.click()
    assert session.state['actions']==[] and v3_play_nested_equal(session.state['rng'],initial_rng)
    try:restored.new_game()
    except RuntimeError as error:assert 'Another widget saved' in str(error)
    else:raise AssertionError('Stale writer was not rejected')
    restored.reload();assert restored.state['game']==session.state['game']
    widget.pass_button.click()
    assert session.state['actions'][0]==81 and all(r['legal'] for r in session.state['turn_records'])
    # An untrained player is not required to choose a winning pass. The terminal-rule fixture
    # applies two exact legal passes directly, with explicit fixture actors, then exercises save/render/reload.
    session.new_game()
    with checkpoint_lock(session.directory):
        state=session._current_locked()
        assert state['config']['provenance']['kind']=='untrained_wiring_fixture'
        board=from_dict(state['game'])
        for _ in range(2):
            action=board.pass_action;before=capture_rng();child=play(board,action)
            assert arena_reference_move(board,action)==child.board
            state['actions'].append(action);state['turn_records'].append({'actor':'exact_terminal_fixture','action':action,'rng_before':before,'legal':True})
            board=child
        state['game']=to_dict(board);session._commit_locked(state)
    widget.render()
    assert session.board.terminal and session.state['actions']==[81,81]
    assert widget.pass_button.disabled and widget.ai_button.disabled and 'White wins by 7.5' in widget.status.value
    assert area_score(session.board)==-7.5
    terminal=V3PlaySession(project,'UNTRAINED_WIRING_FIXTURE',session_id,fixture_model=copy.deepcopy(model))
    assert terminal.board.terminal and terminal.state['game']==session.state['game']
    manifest=v3_read_json(session.directory/'manifest.json')
    assert len(manifest['entries'])>=2 and manifest['latest'] and manifest['best']
    report={'schema':V3_PLAY_SCHEMA,'status':'PASS','fixture_provenance':'UNTRAINED_WIRING_FIXTURE',
        'training_performed':False,'strength_claim':False,'session_id':session_id,'sources':v3_play_sources(project),
        'widget_board_click':True,'legal_ai_reply':True,'queen_cycles':2,'inference_budget_s':1.,
        'full_history_reload':True,'exact_rng_reload':True,'undo_pair_and_rng':True,'stale_writer_rejected':True,
        'two_pass_terminal_verified':True,'terminal_fixture':'Two exact legal passes, explicitly labelled fixture actors; not learned move choices','terminal_black_area_score':-7.5,'latest_two_and_best':True,
        'selected_baseline_unchanged':True,'network_service_started':False}
    atomic_json(v3_report_path(project,'play_validation.json'),report)
    return report,widget

def v3_trained_play_validation(project,run_id,*,device='cpu',session_id=None):
    """An isolated real-candidate integration check; never consumes or resets a user's game."""
    project=Path(project);source_identity=v3_source_identity(project)
    play_path=project/V3_ROOT/'12_play_predictive_ahr.ipynb'
    play_code_sha256=v3_code_sha(play_path)
    reference_model,metadata=v3_load_model(project,run_id,device=device,selection='latest')
    if metadata['provenance'].get('purpose')!='go_training' or metadata['config'].get('validation_force_pass',False) or metadata['optimizer_steps']<1:
        raise ValueError('Integration requires a genuine trained Go candidate, not a synthetic fixture')
    if metadata['phase']!='complete':raise ValueError('Use a completed pilot or completed training run for frozen integration validation')
    reference_digest=arena_tensor_digest(reference_model.state_dict())
    session_id=session_id or 'v3play-trained-check-'+uuid.uuid4().hex[:12]
    if not session_id.startswith('v3play-trained-check-'):raise ValueError('Use an isolated v3play-trained-check- session ID')
    if (project/'checkpoints'/session_id/'manifest.json').exists():raise ValueError('Validation needs a fresh isolated session; omit session_id to create one')
    session=V3PlaySession(project,run_id,session_id,device=device)
    provenance=session.identity['provenance']
    if provenance['kind']!='trained_v3_candidate' or provenance['checkpoint_sha256']!=metadata['checkpoint_sha256']:
        raise RuntimeError('Candidate checkpoint changed between loading and pinning')
    assert session.identity['model_weights_sha256']==reference_digest
    widget=V3PlayWidget(session)
    widget._buttons[40].click()
    if len(session.state['actions'])!=2 or session.state['actions'][0]!=40:
        raise AssertionError('Trained candidate did not complete a human click and legal reply: '+str(session.state['last_error']))
    assert all(record['legal'] for record in session.state['turn_records'])
    decision=session.state['last_decision']
    assert decision['status']=='LEGAL_MOVE' and decision['queen_cycles']==2 and decision['budget_s']==1.
    assert decision['elapsed_s']<=1. and decision['latent_guidance'] is False
    assert to_dict(v3_play_reconstruct(session.state['actions']))==session.state['game']
    saved=copy.deepcopy(session.state)
    # Resume takes the complete pinned model from the play checkpoint, preserving the original training-checkpoint identity.
    restored=V3PlaySession(project,run_id,session_id,device=device)
    assert restored.identity['provenance']['checkpoint_sha256']==metadata['checkpoint_sha256']
    assert arena_tensor_digest(restored.model.state_dict())==reference_digest
    assert v3_play_nested_equal(restored.state,saved)
    assert v3_play_nested_equal(capture_rng(),saved['rng'])
    manifest=v3_read_json(session.directory/'manifest.json')
    assert len(manifest['entries'])>=2 and manifest['latest'] and manifest['best']
    if source_identity!=v3_source_identity(project) or play_code_sha256!=v3_code_sha(play_path):
        raise RuntimeError('Definitions changed during trained play validation; rerun after source freeze')
    report={'schema':V3_PLAY_SCHEMA,'status':'PASS','run_id':run_id,'session_id':session_id,
        'checkpoint_sha256':metadata['checkpoint_sha256'],'model_weights_sha256':reference_digest,
        'source_identity':source_identity,'play_code_sha256':play_code_sha256,
        'training_purpose':metadata['provenance']['purpose'],'trained_optimizer_steps':metadata['optimizer_steps'],
        'candidate_not_promoted':True,'fixture_model_used':False,'human_board_click':True,'real_ai_legal_reply':True,
        'actions':saved['actions'],'decision':decision,'pinned_checkpoint_reload':True,'exact_game_reload':True,
        'exact_full_rng_reload':True,'latest_two_and_best':True,'selected_baseline_unchanged':True,
        'network_service_started':False,'new_training_performed':False,
        'interaction_scope':'Python ipywidgets callback dispatch; not a live browser communication test'}
    atomic_json(v3_report_path(project,'trained_play_validation.json'),report)
    return report


In [ ]:
V3_PLAY_VALIDATION,V3_PLAY_FIXTURE_WIDGET=v3_play_smoke(PROJECT)
print(json.dumps({k:v for k,v in V3_PLAY_VALIDATION.items() if k!='sources'},indent=2))

## Optional trained-candidate integration

After a genuine V3 pilot or training run is complete, call `v3_trained_play_validation(PROJECT, run_id)`, or set `AHRV3_PLAY_VALIDATE_RUN` before Run All. This creates an isolated validation game, clicks the board, obtains a real AHR reply, and verifies exact game/RNG/model reload. It writes `reports/ahr_jepa_v3/trained_play_validation.json` with the actual checkpoint and source hashes. Synthetic or unfinished runs are rejected. This check does not train, promote a model or change an existing game.

In [ ]:
V3_VALIDATE_TRAINED_RUN=os.environ.get('AHRV3_PLAY_VALIDATE_RUN','').strip()
if V3_VALIDATE_TRAINED_RUN:
    V3_TRAINED_PLAY_VALIDATION=v3_trained_play_validation(PROJECT,V3_VALIDATE_TRAINED_RUN,device=os.environ.get('AHRV3_PLAY_DEVICE','cpu'))
    print(json.dumps({k:v for k,v in V3_TRAINED_PLAY_VALIDATION.items() if k!='source_identity'},indent=2))
else:
    print('Trained-candidate integration is ready; no completed training run was selected for this optional check.')

## Open an explicitly chosen candidate

Run this notebook with `AHRV3_PLAY_RUN` set to a trained V3 run. The default session name includes the run, and its manifest pins the exact checkpoint. If the trainer produces a later checkpoint, use a new session ID to inspect that newer model. The current game reloads its pinned model weights from the validated play checkpoint and keeps its original training-checkpoint identity.

In [ ]:
V3_PLAY_RUN=os.environ.get('AHRV3_PLAY_RUN','').strip()
V3_PLAY_SESSION=os.environ.get('AHRV3_PLAY_SESSION','v3play-'+V3_PLAY_RUN).strip()
if V3_PLAY_RUN:
    V3_PLAY_SESSION_OBJECT=V3PlaySession(PROJECT,V3_PLAY_RUN,V3_PLAY_SESSION,device=os.environ.get('AHRV3_PLAY_DEVICE','cpu'))
    V3_PLAY_WIDGET=V3PlayWidget(V3_PLAY_SESSION_OBJECT)
    V3_PLAY_WIDGET.display()
else:
    display(HTML('<div style="padding:14px;border:1px solid #8899aa;border-radius:5px"><b>Ready for a trained V3 candidate.</b><br>No candidate run was selected. Set <code>AHRV3_PLAY_RUN</code>, restart the kernel, and Run All. The successful checks above used an explicitly untrained fixture. The original selected player remains available in the original notebook 12.</div>'))